In [ ]:
from google.colab import drive
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import Dataset, DataLoader
from torch import optim
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import copy
from tqdm import tqdm
from torch.utils.data import WeightedRandomSampler
import torchvision.transforms as transforms
from PIL import Image
import tarfile
import os
import glob
from sklearn.model_selection import train_test_split
import cv2
from scipy.ndimage import binary_fill_holes
import json
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from torchvision.ops import SqueezeExcitation
from sklearn.utils.class_weight import compute_class_weight
from scipy.stats import ttest_rel, wilcoxon
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
device = "cuda:0"

# **Data Loading**

In [ ]:
drive.mount('/content/drive')

In [ ]:
train_img_dir = '/content/drive/MyDrive/COMP448_Group_Assignments/Term_Project/data/train/img'
train_ann_dir = '/content/drive/MyDrive/COMP448_Group_Assignments/Term_Project/data/train/ann'

test_img_dir = '/content/drive/MyDrive/COMP448_Group_Assignments/Term_Project/data/test/img'
test_ann_dir = '/content/drive/MyDrive/COMP448_Group_Assignments/Term_Project/data/test/ann'

In [ ]:
def extract_data(img_dir, ann_dir):

  data = []

  for fname in os.listdir(ann_dir):
    ann_path = os.path.join(ann_dir, fname)

    with open(ann_path, 'r') as f:
      ann = json.load(f)

      # Find the class from annotation data - benign or malignant
      tags = ann.get("tags", [])
      label = None
      for tag in tags:
          if tag["name"] in ["benign", "malignant", "benign_without_callback"]:
              label = tag["name"]
              break

      # If label is found, match the corresponding label to the image
      if label:
          image_name = fname.replace(".json", "")
          image_path = os.path.join(img_dir, image_name)
          data.append((image_path, label))


  return pd.DataFrame(data, columns=["image_path", "label"])


In [ ]:
train_df = extract_data(train_img_dir, train_ann_dir)
test_df = extract_data(test_img_dir, test_ann_dir)


In [ ]:
display(train_df.head())
display(test_df.head())

# **Class Distribution**


In [ ]:
classDict = {
    "benign": 0,
    "malignant": 1,
    "benign_without_callback": 2
}


In [ ]:
# Count label frequencies
train_counts = train_df['label'].value_counts()
test_counts = test_df['label'].value_counts()


fig, axs = plt.subplots(1, 2, figsize=(10, 4))

axs[0].bar(train_counts.index, train_counts.values, color='blue')
axs[0].set_title(f"Train Set (Size={len(train_df)})")
axs[0].set_ylabel("Count")

axs[1].bar(test_counts.index, test_counts.values, color='green')
axs[1].set_title(f"Test Set (Size={len(test_df)})")
axs[1].set_ylabel("Count")

plt.tight_layout()
plt.show()

# **Dataset**

In [ ]:
class DDSM_Dataset(Dataset):
    def __init__(self, dataframe, classDict, augment=False, resize=(224, 224)):
        self.data = dataframe.reset_index(drop=True)
        self.classDict = classDict
        self.augment = augment
        self.resize = resize

        # Define transforms
        self.train_transform = transforms.Compose([
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(10),
            transforms.Resize(self.resize),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5], std=[0.5])
        ])
        self.val_transform = transforms.Compose([
            transforms.Resize(self.resize),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5], std=[0.5])
        ])

    def __len__(self):
        return len(self.data)

    def transform(self, img):

        # Convert to PIL and apply transforms
        pil_img = Image.fromarray(img).convert('L')
        if self.augment:
            return self.train_transform(pil_img)
        else:
            return self.val_transform(pil_img)

    def __getitem__(self, idx):
        image_path = self.data.loc[idx, "image_path"]
        label_str = self.data.loc[idx, "label"]

        # Load grayscale image from file
        img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            raise ValueError(f"Could not read image at {image_path}")

        img = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX).astype('uint8')
        img_tensor = self.transform(img)

        label = self.classDict[label_str]
        label = torch.tensor(label, dtype=torch.long)

        return img_tensor, label


In [ ]:
# For SwinUNet Model
class DDSM_Dataset_RGB(Dataset):
    def __init__(self, dataframe, classDict, augment=False, resize=(224, 224)):
        self.data = dataframe.reset_index(drop=True)
        self.classDict = classDict
        self.augment = augment
        self.resize = resize

        self.train_transform = transforms.Compose([
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(degrees=10),
            transforms.RandomResizedCrop(self.resize, scale=(0.8, 1.0)),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5]*3, std=[0.5]*3)
        ])

        self.val_transform = transforms.Compose([
            transforms.Resize(self.resize),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5]*3, std=[0.5]*3)
        ])

    def __len__(self):
        return len(self.data)

    def transform(self, img):
        pil_img = Image.fromarray(img).convert('RGB')
        if self.augment:
            return self.train_transform(pil_img)
        else:
            return self.val_transform(pil_img)

    def __getitem__(self, idx):
        image_path = self.data.loc[idx, "image_path"]
        label_str = self.data.loc[idx, "label"]

        img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            raise ValueError(f"Could not read image at {image_path}")

        img = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX).astype('uint8')
        img_tensor = self.transform(img)

        label = self.classDict[label_str]
        label = torch.tensor(label, dtype=torch.long)

        return img_tensor, label


In [ ]:
X_train, X_val = train_test_split(train_df, test_size=0.2, stratify=train_df['label'], random_state=42)

train_dataset = DDSM_Dataset(X_train, classDict, augment=True)
val_dataset   = DDSM_Dataset(X_val, classDict, augment=False)
test_dataset  = DDSM_Dataset(test_df, classDict, augment=False)

In [ ]:
# Compute class weights
train_labels = X_train['label'].map(classDict).values
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels),
    y=train_labels
)

class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)

In [ ]:
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size, num_workers=2, shuffle=True, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size, num_workers=2, pin_memory=True)

# Calculating class weights
"""
train_labels = [classDict[label] for label in X_train['label']]
train_labels = torch.tensor(train_labels)

class_sample_count = torch.tensor([(train_labels == t).sum() for t in torch.unique(train_labels, sorted=True)])
weight = 1.0 / class_sample_count.float()
class_weights = weight[train_labels]

sampler = WeightedRandomSampler(class_weights, len(class_weights), replacement=True)
train_loader = DataLoader(train_dataset, batch_size=batch_size, sampler=sampler, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, num_workers=2, pin_memory=True)
"""

dataloaders = {
    'train': train_loader,
    'val': val_loader
}


**Visualize Random Samples**

In [ ]:
import random

fig, axs = plt.subplots(1, 3, figsize=(12, 4))

for i in range(3):
    idx = random.randint(0, len(train_dataset) - 1)
    img_tensor, label = train_dataset[idx]

    # Convert to numpy and denormalize
    img_np = img_tensor.squeeze().numpy()  # (H, W)
    img_np = (img_np * 0.5) + 0.5  # from [-1, 1] to [0, 1]

    axs[i].imshow(img_np, cmap='gray')
    axs[i].set_title(f"Label: {list(classDict.keys())[label.item()]}")
    axs[i].axis('off')

plt.tight_layout()
plt.show()

# **Models**

## **Simple CNN**

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=3):
        super(SimpleCNN, self).__init__()

        # Convolutional Block 1
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.pool1 = nn.MaxPool2d(2, 2)

        # Convolutional Block 2
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.pool2 = nn.MaxPool2d(2, 2)

        # Convolutional Block 3
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.pool3 = nn.MaxPool2d(2, 2)

        # Global Average Pooling
        self.gap = nn.AdaptiveAvgPool2d((1, 1))

        # Fully Connected Layers
        self.fc1 = nn.Linear(128, 64)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.pool1(F.relu(self.bn1(self.conv1(x))))
        x = self.pool2(F.relu(self.bn2(self.conv2(x))))
        x = self.pool3(F.relu(self.bn3(self.conv3(x))))
        x = self.gap(x)
        x = x.reshape(x.size(0), -1)  # Flatten
        x = self.dropout(F.relu(self.fc1(x)))
        x = self.fc2(x)
        return x

In [ ]:
model_cnn = SimpleCNN()
model_cnn.to(device=device)

## **ResNet50**

#### Configuration 1

In [ ]:
class ResNet50_1(nn.Module):
    def __init__(self, in_channels=1, num_classes=3, base_channels=16, heads=4):
        super().__init__()
        self.resnet50_model = models.resnet50(weights='IMAGENET1K_V1')

        init_conv1 = self.resnet50_model.conv1
        w = init_conv1.weight.data
        self.resnet50_model.conv1 = nn.Conv2d(
            in_channels=in_channels,
            out_channels=self.resnet50_model.conv1.out_channels,
            kernel_size=self.resnet50_model.conv1.kernel_size,
            stride=self.resnet50_model.conv1.stride,
            padding=self.resnet50_model.conv1.padding,
            bias=False
        )

        self.resnet50_model.conv1.weight.data = w.mean(dim=1, keepdim=True)

        in_features = self.resnet50_model.fc.in_features
        self.resnet50_model.fc = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(in_features, num_classes)
        )

    def forward(self, x):
        return self.resnet50_model(x)

#### Configuration 2

In [ ]:
class ResNet50_2(ResNet50_1):
    def __init__(self, in_channels=1, num_classes=3):
        super().__init__(in_channels, num_classes)
        for name, param in self.resnet50_model.named_parameters():
            if not (name.startswith("layer4") or name.startswith("fc")):
                param.requires_grad = False

#### Configuration 3

In [ ]:
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(channels, channels // reduction, 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels // reduction, channels, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        w = self.avgpool(x)
        w = self.fc(w)
        return x * w

In [ ]:
class ResNet50_3(ResNet50_1):
    def __init__(self, in_channels=1, num_classes=3):
        super().__init__(in_channels, num_classes)

        original_layer3 = self.resnet50_model.layer3
        layer3_out = original_layer3[-1].conv3.out_channels
        self.resnet50_model.layer3 = nn.Sequential(
            original_layer3,
            SEBlock(channels=layer3_out, reduction=16)
        )

        original_layer4 = self.resnet50_model.layer4
        layer4_out = original_layer4[-1].conv3.out_channels
        self.resnet50_model.layer4 = nn.Sequential(
            original_layer4,
            SEBlock(channels=layer4_out, reduction=16)
        )

    def forward(self, x):
        return self.resnet50_model(x)

#### Configuration 4

In [ ]:
class ResNet50_4(ResNet50_1):
    def __init__(self, in_channels=1, num_classes=3):
        super().__init__(in_channels, num_classes)

        original_in_featuress = self.resnet50_model.fc[1].in_features

        self.resnet50_model.fc = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(original_in_featuress, 1024),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(1024, num_classes)
        )

    def forward(self, x):
        return self.resnet50_model(x)

In [ ]:
model_resnet1 = ResNet50_1(in_channels=1, num_classes=3).to(device)
model_resnet2 = ResNet50_2(in_channels=1, num_classes=3).to(device)
model_resnet3 = ResNet50_3(in_channels=1, num_classes=3).to(device)
model_resnet4 = ResNet50_4(in_channels=1, num_classes=3).to(device)

## **U-Net**

#### Configuration 1

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.bn1   = nn.BatchNorm2d(out_channels)
        self.relu1 = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.bn2   = nn.BatchNorm2d(out_channels)
        self.relu2 = nn.ReLU(inplace=True)

    def forward(self, x):
        x = self.relu1(self.bn1(self.conv1(x)))
        x = self.relu2(self.bn2(self.conv2(x)))
        return x


class UpBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.up   = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
        self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x1, x2):
        x1 = self.up(x1)
        diffW = x2.size(3) - x1.size(3)
        diffH = x2.size(2) - x1.size(2)
        x1 = F.pad(x1, (diffW//2, diffW-diffW//2, diffH//2, diffH-diffH//2))
        x  = torch.cat([x2, x1], dim=1)
        return self.conv(x)


class DownBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.pool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels)
        )
    def forward(self, x):
        return self.pool_conv(x)


class UNetClassifier(nn.Module):
    def __init__(self, in_channels, num_classes):
        super().__init__()
        self.enc1   = DoubleConv(in_channels,  16)
        self.enc2   = DownBlock(16,  32)
        self.enc3   = DownBlock(32,  64)
        self.enc4   = DownBlock(64, 128)
        self.center = DownBlock(128,256)
        self.dec4 = UpBlock(256,128)
        self.dec3 = UpBlock(128,64)
        self.dec2 = UpBlock(64,32)
        self.dec1 = UpBlock(32,16)
        self.final = nn.Conv2d(16, 64, kernel_size=1)
        self.gap = nn.AdaptiveAvgPool2d((1,1))
        self.classifier = nn.Sequential(
            nn.BatchNorm1d(64),
            nn.Dropout(0.5),
            nn.Linear(64, 512),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(512),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x1 = self.enc1(x)
        x2 = self.enc2(x1)
        x3 = self.enc3(x2)
        x4 = self.enc4(x3)
        xc = self.center(x4)
        x  = self.dec4(xc, x4)
        x  = self.dec3(x,  x3)
        x  = self.dec2(x,  x2)
        x  = self.dec1(x,  x1)
        x  = self.final(x)
        x  = self.gap(x)
        x  = x.view(x.size(0), -1)
        return self.classifier(x)


model_unet = UNetClassifier(in_channels=1, num_classes=3).to("cuda")

#### Configuration 2

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.act1  = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.act2  = nn.ReLU(inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.act1(self.conv1(x))
        x = self.act2(self.conv2(x))
        return x

class Down(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.pool = nn.MaxPool2d(2)
        self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x):
        return self.conv(self.pool(x))

class UNetClassifier(nn.Module):
    def __init__(
        self,
        in_channels: int = 1,
        num_classes: int = 3,
        initial_features: int = 16,
        p_dropout: float = 0.1
    ):
        super().__init__()

        # encoder
        self.inc   = DoubleConv(in_channels, initial_features)
        self.down1 = Down(initial_features,   initial_features * 2)
        self.down2 = Down(initial_features*2, initial_features * 4)
        self.down3 = Down(initial_features*4, initial_features * 8)
        self.down4 = Down(initial_features*8, initial_features * 16)

        self.drop = nn.Dropout2d(p_dropout)
        self.gap = nn.AdaptiveAvgPool2d((1,1))
        C = initial_features * 16

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.BatchNorm1d(C),
            nn.Dropout(p_dropout),
            nn.Linear(C, C//2, bias=False),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(C//2),
            nn.Dropout(p_dropout),
            nn.Linear(C//2, num_classes)
        )

    def forward(self, x):
        x = self.inc(x)
        x = self.down1(x)
        x = self.down2(x)
        x = self.down3(x)
        x = self.down4(x)

        x = self.drop(x)
        x = self.gap(x)
        return self.classifier(x)

model_unet = UNetClassifier(in_channels=1, num_classes=3, initial_features=16,p_dropout=0.2).to(device)

## **Swin-Unet**

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class PatchEmbedding(nn.Module):
    def __init__(self, img_size=224, patch_size=4, in_channels=3, embed_dim=96):
        super().__init__()
        self.proj = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x):
        x = self.proj(x)
        B, C, H, W = x.shape
        x = x.flatten(2).transpose(1, 2)
        x = self.norm(x)
        return x, H, W


class WindowSelfAttention(nn.Module):
    def __init__(self, dim, window_size, num_heads):
        super().__init__()
        self.window_size = window_size
        self.attn = nn.MultiheadAttention(embed_dim=dim, num_heads=num_heads, batch_first=True)
        self.norm = nn.LayerNorm(dim)

    def forward(self, x, H, W):
        B, N, C = x.shape
        x = x.transpose(1, 2).view(B, C, H, W)
        x = x.unfold(2, self.window_size, self.window_size).unfold(3, self.window_size, self.window_size)
        B, C, wh, ww, wh_size, ww_size = x.shape
        x = x.contiguous().view(B * wh * ww, C, wh_size * ww_size).transpose(1, 2)
        x, _ = self.attn(x, x, x)
        x = self.norm(x)
        x = x.transpose(1, 2).contiguous().view(B, wh, ww, C, wh_size, ww_size)
        x = x.permute(0, 3, 1, 4, 2, 5).contiguous().view(B, C, H, W)
        x = x.flatten(2).transpose(1, 2)
        return x


class SwinBlock(nn.Module):
    def __init__(self, dim, window_size, num_heads):
        super().__init__()
        self.attn = WindowSelfAttention(dim, window_size, num_heads)
        self.mlp = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, dim * 4),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(dim * 4, dim),
            nn.Dropout(0.1)
        )

    def forward(self, x, H, W):
        x = self.attn(x, H, W) + x
        x = self.mlp(x) + x
        return x

class SwinTransformer(nn.Module):
    def __init__(self, img_size=224, patch_size=4, in_channels=3, embed_dim=96,
                 depths=2, num_heads=4, window_size=7, num_classes=3):
        super().__init__()
        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, embed_dim)
        self.pos_embed = nn.Parameter(torch.zeros(1, (img_size // patch_size)**2, embed_dim))
        self.layers = nn.ModuleList([
            SwinBlock(embed_dim, window_size, num_heads)
            for _ in range(depths)
        ])
        self.norm = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(0.2)
        self.head = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        x, H, W = self.patch_embed(x)
        x = x + self.pos_embed
        for blk in self.layers:
            x = blk(x, H, W)
        x = self.norm(x)
        x = self.dropout(x.mean(dim=1))  # Global average pooling
        return self.head(x)


class SwinWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        if x.shape[1] == 1:
            x = x.repeat(1, 3, 1, 1)
        return self.model(x)

base_model_swin = SwinTransformer(img_size=224, patch_size=4, in_channels=3,
                                  embed_dim=96, depths=2, num_heads=4,
                                  window_size=7, num_classes=3)

model_swin = SwinWrapper(base_model_swin).to("cuda")





# **Training**

## **Hyperparameters**

In [ ]:
# Cross entropy loss for classification
calc_loss =  nn.CrossEntropyLoss()

# Focal Loss
"""
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=0.5, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none', weight=self.alpha)
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

calc_loss_unet = FocalLoss(alpha=weights, gamma=0.5)
"""

# Weighted Cross Entropy
"""
calc_loss = nn.CrossEntropyLoss(weight=class_weights)
"""

#### **CNN Hyperparameters**

In [ ]:
# Hyperparameters
cnn_lrRate = 0.0001
cnn_weightDecay = 1e-3
cnn_numberOfEpoch = 30
cnn_early_stop = 10

# Loss function
calc_loss_cnn = nn.CrossEntropyLoss(weight=class_weights)

# Optimizers
optimizer_cnn = optim.Adam(model_cnn.parameters(), lr=cnn_lrRate, weight_decay=cnn_weightDecay)

# Schedulers
scheduler_cnn = optim.lr_scheduler.ReduceLROnPlateau(optimizer_cnn, mode='min', patience=3, factor=0.5)

#### **ResNet50 Hyperparameters**


In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2, weight=None):
        super().__init__()
        self.gamma = gamma
        self.cross_entropy = nn.CrossEntropyLoss(weight=weight)

    def forward(self, input, target):
        log_p = nn.functional.log_softmax(input, dim=1)
        p = torch.exp(log_p)
        focal_term = (1 - p).pow(self.gamma)
        loss = self.cross_entropy(input, target) * focal_term.gather(1, target.unsqueeze(1)).squeeze()
        return loss.mean()

In [ ]:
# Hyperparameters
resnet_lrRate = 0.0001
resnet_weightDecay = 1e-3
resnet_numberOfEpoch = 30
resnet_early_stop = 10

# Optimizers
optimizer_resnet = optim.Adam(model_resnet1.parameters(), lr=resnet_lrRate, weight_decay=resnet_weightDecay)

#calc_loss_resnet = FocalLoss(gamma=2.0, weight=class_weights.to(device))
calc_loss_resnet = nn.CrossEntropyLoss(weight=class_weights.to(device))

# Schedulers
scheduler_resnet = optim.lr_scheduler.ReduceLROnPlateau(optimizer_resnet, mode='min', patience=3, factor=0.5)

#### **U-Net Hyperparameters**

In [ ]:
# Hyperparameters
unet_lrRate = 1e-5
#unet_lrRate = 1e-4
unet_weightDecay = 1e-3
unet_numberOfEpoch = 20
unet_early_stop = 5


optimizer_unet = optim.AdamW(
    filter(lambda p: p.requires_grad, model_unet.parameters()),
    lr=1e-3, weight_decay=1e-2
)

scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer_unet,
    max_lr=1e-3,
    steps_per_epoch=len(train_loader),
    epochs=10,
    pct_start=0.2,
    anneal_strategy="cos"
)
"""
optimizer_unet = optim.Adam(model_unet.parameters(), lr=unet_lrRate, weight_decay=unet_weightDecay)
scheduler_unet = optim.lr_scheduler.ReduceLROnPlateau(optimizer_unet, mode='min', patience=3, factor=0.5)
"""

#### **Swin Hyperparameters**

In [ ]:
# Hyperparameters
swin_lrRate = 0.0001
swin_weightDecay = 1e-3
swin_numberOfEpoch = 30
swin_early_stop = 10

# Optimizers
optimizer_swin = optim.Adam(model_swin.parameters(), lr=swin_lrRate, weight_decay=swin_weightDecay)

# Schedulers
scheduler_swin = optim.lr_scheduler.ReduceLROnPlateau(optimizer_swin, mode='min', patience=3, factor=0.5)

train_dataset_swin = DDSM_Dataset_RGB(X_train, classDict, augment=True)
val_dataset_swin   = DDSM_Dataset_RGB(X_val, classDict, augment=False)

train_loader_swin = DataLoader(train_dataset_swin, batch_size=32, sampler=sampler, num_workers=2, pin_memory=True)
val_loader_swin   = DataLoader(val_dataset_swin, batch_size=32, num_workers=2, pin_memory=True)

dataloaders_swin = {
    'train': train_loader_swin,
    'val': val_loader_swin
}


## **Train**

In [ ]:
from sklearn.metrics import accuracy_score

def train(model, dataloaders, calc_loss, optimizer, scheduler, numberOfEpoch, early_stop, device):

    best_model = copy.deepcopy(model.state_dict())
    best_f1 = 0.0
    early_stop_counter = 0

    performance_metrics = {
        'train_loss': [],
        'train_f1': [],
        'train_precision': [],
        'train_recall': [],
        'train_accuracy': [],
        'val_loss': [],
        'val_f1': [],
        'val_precision': [],
        'val_recall': [],
        'val_accuracy': []
    }

    for epoch in range(numberOfEpoch):

        print(f"\nEpoch {epoch+1}")
        epoch_val_loss = 0.0

        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
                context = torch.enable_grad
            else:
                model.eval()
                context = torch.no_grad

            total_loss = 0.0
            y_preds = []
            y_trues = []

            with context():
                with tqdm(dataloaders[phase], unit="batch") as tbar:
                    for images, labels in tbar:
                        tbar.set_description(f"{phase.capitalize()} Epoch {epoch+1}")

                        images = images.to(device)
                        labels = labels.to(device)

                        optimizer.zero_grad()
                        outputs = model(images)
                        loss = calc_loss(outputs, labels)

                        if phase == 'train':
                            loss.backward()
                            optimizer.step()

                        total_loss += loss.item() * images.size(0)
                        preds = torch.argmax(outputs, dim=1)

                        y_preds.extend(preds.cpu().numpy())
                        y_trues.extend(labels.cpu().numpy())

            epoch_loss = total_loss / len(dataloaders[phase].dataset)
            f1 = f1_score(y_trues, y_preds, average='macro', zero_division=0)
            precision = precision_score(y_trues, y_preds, average='macro', zero_division=0)
            recall = recall_score(y_trues, y_preds, average='macro', zero_division=0)
            accuracy = accuracy_score(y_trues, y_preds)

            print(f"{phase.capitalize()} Loss: {epoch_loss:.4f} | F1: {f1:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f} | Accuracy: {accuracy:.4f}")

            performance_metrics[f"{phase}_loss"].append(epoch_loss)
            performance_metrics[f"{phase}_f1"].append(f1)
            performance_metrics[f"{phase}_precision"].append(precision)
            performance_metrics[f"{phase}_recall"].append(recall)
            performance_metrics[f"{phase}_accuracy"].append(accuracy)

            if phase == 'val':
                epoch_val_loss = epoch_loss
                if f1 > best_f1:
                    best_f1 = f1
                    best_model = copy.deepcopy(model.state_dict())
                    early_stop_counter = 0
                else:
                    early_stop_counter += 1

        scheduler.step(epoch_val_loss)

        if early_stop_counter >= early_stop:
            print(f"Early stopping at epoch {epoch+1}")
            model.load_state_dict(best_model)
            return best_model, performance_metrics

    model.load_state_dict(best_model)
    return best_model, performance_metrics

## **Parameter Selection**

In [ ]:
best_model_cnn, performance_metrics_cnn = train(model_cnn, dataloaders, calc_loss_cnn, optimizer_cnn, scheduler_cnn, cnn_numberOfEpoch, cnn_early_stop, device)
best_model_unet, performance_metrics_unet = train( model_unet, dataloaders, calc_loss, optimizer_unet, unet_numberOfEpoch, unet_early_stop, device)
best_model_swin, performance_metrics = train(model_swin, dataloaders_swin, calc_loss, optimizer_swin, scheduler_swin, swin_numberOfEpoch, swin_early_stop, device)
best_model_resnet, performance_metrics_resnet = train(model_resnet1, dataloaders, calc_loss, optimizer_resnet, scheduler_resnet, 1, resnet_early_stop, device)


### K-Fold Cross Validation

In [ ]:
def run_kfold(model_class, df, classDict, lrRate, weightDecay, numberOfEpoch, early_stop, num_classes=3, k=5, device="cuda:0"):
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)

    fold_metrics = []
    fold_models = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(df, df['label'])):
        print(f"\n--- Fold {fold+1}/{k} ---")
        train_df_fold = df.iloc[train_idx]
        val_df_fold = df.iloc[val_idx]

        train_dataset = DDSM_Dataset(train_df_fold, classDict, augment=True)
        val_dataset = DDSM_Dataset(val_df_fold, classDict, augment=False)

        dataloaders = {
            'train': DataLoader(train_dataset, batch_size=32, shuffle=True),
            'val': DataLoader(val_dataset, batch_size=32, shuffle=False)
        }

        model = model_class(num_classes=num_classes).to(device)
        optimizer = optim.Adam(model.parameters(), lr=lrRate, weight_decay=weightDecay)
        criterion = nn.CrossEntropyLoss()

        best_model, performance_metrics = train(
            model=model,
            dataloaders=dataloaders,
            calc_loss=criterion,
            optimizer=optimizer,
            scheduler=optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3),
            numberOfEpoch=numberOfEpoch,
            early_stop=early_stop,
            device=device
        )

        fold_models.append(copy.deepcopy(best_model))

        fold_metrics.append({
            "fold": fold + 1,
            "train_loss": performance_metrics['train_loss'][-1],
            "train_f1": performance_metrics['train_f1'][-1],
            "train_precision": performance_metrics['train_precision'][-1],
            "train_recall": performance_metrics['train_recall'][-1],
            "train_accuracy": performance_metrics['train_accuracy'][-1],
            "val_loss": performance_metrics['val_loss'][-1],
            "val_f1": performance_metrics['val_f1'][-1],
            "val_precision": performance_metrics['val_precision'][-1],
            "val_recall": performance_metrics['val_recall'][-1],
            "val_accuracy": performance_metrics['val_accuracy'][-1]
        })

    df_metrics = pd.DataFrame(fold_metrics)
    print("\n--- Overall K-Fold Results ---")
    print(df_metrics.mean(numeric_only=True))

    return df_metrics, fold_models

In [ ]:
results_cnn, fold_models_cnn = run_kfold(lambda num_classes: SimpleCNN(), train_df, classDict, lrRate=cnn_lrRate, weightDecay=cnn_weightDecay, numberOfEpoch=cnn_numberOfEpoch, early_stop=cnn_early_stop, device=device)
results_unet, fold_models_unet = run_kfold(lambda num_classes: UNetClassifier(num_classes), train_df, classDict, lrRate=unet_lrRate, weightDecay=unet_weightDecay, numberOfEpoch=unet_numberOfEpoch, early_stop=unet_early_stop, device=device)
results_resnet, fold_models_resnet = run_kfold(lambda num_classes: resnet(), train_df, classDict, lrRate=resnet_lrRate, weightDecay=resnet_weightDecay, numberOfEpoch=resnet_numberOfEpoch, early_stop=resnet_early_stop, device=device)
result_swin, fold_models_swin = run_kfold(lambda num_classes: SwinTransformer(num_classes), train_df, classDict, lrRate=swin_lrRate, weightDecay=swin_weightDecay, numberOfEpoch=swin_numberOfEpoch, early_stop=swin_early_stop, device=device

### Save Model and Metrics

In [ ]:
def save_model_and_metrics(model, performance_metrics, model_filename="model.pth", metrics_filename="performance_metrics.json"):

    torch.save(model.state_dict(), model_filename)
    performance_metrics = {k: [float(v) for v in vals] for k, vals in performance_metrics.items()}
    with open(metrics_filename, 'w') as f:
        json.dump(performance_metrics, f, indent=2)

In [ ]:
save_model_and_metrics(model_cnn, performance_metrics_cnn, model_filename="model_cnn.pth", metrics_filename="performance_metrics_cnn.json")

### Statistical Testing

In [ ]:
for metric in ["f1", "accuracy", "precision", "recall"]:
    t_stat, t_p = ttest_rel(results_unet[metric], results_cnn[metric])
    print(f"T-Test on {metric.capitalize()}: t = {t_stat:.4f}, p = {t_p:.4f}")

    w_stat, w_p = wilcoxon(results_unet[metric], results_cnn[metric])
    print(f"Wilcoxon on {metric.capitalize()}: W = {w_stat}, p = {w_p:.4f}")

## **Visualization**

### **Fold-wise Performance Graphs**


In [ ]:
def plot_fold_metrics(results_df, model_name="Model"):
    import matplotlib.pyplot as plt
    plt.figure(figsize=(8,5))
    plt.plot(results_df['fold'], results_df['val_f1'], marker='o', label="F1 Score")
    plt.plot(results_df['fold'], results_df['accuracy'], marker='s', label="Accuracy")
    plt.title(f"{model_name} Fold-wise Performance")
    plt.xlabel("Fold")
    plt.ylabel("Score")
    plt.ylim(0, 1)
    plt.grid(True)
    plt.legend()
    plt.show()


plot_fold_metrics(results_unet, "UNet")
plot_fold_metrics(results_cnn, "SimpleCNN")

### **Loss Curve**

In [ ]:
train_losses = performance_metrics_resnet['train_loss']
validation_losses = performance_metrics_resnet['val_loss']

plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Training Loss')
plt.plot(validation_losses, label='Validation Loss')

plt.title('Loss Curve', fontsize=14)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.xticks(range(len(train_losses)))
plt.legend(loc='upper right', fontsize=12)
plt.tight_layout()
plt.show()

### **F1-Score Curve**

In [ ]:
train_f1 = performance_metrics_resnet['train_f1']
validation_f1 = performance_metrics_resnet['val_f1']

plt.figure(figsize=(10, 5))
plt.plot(train_f1, label='Training F1-Score')
plt.plot(validation_f1, label='Validation F1-Score')

plt.title('F1-Score Curve', fontsize=14)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('F1-Score', fontsize=12)
plt.xticks(range(len(train_f1)))
plt.legend(loc='upper right', fontsize=12)
plt.tight_layout()
plt.show()

### **Sensitivity Curve**

In [ ]:
train_sensitivity = performance_metrics_resnet['train_recall']
validation_sensitivity = performance_metrics_resnet['val_recall']

plt.figure(figsize=(10, 5))
plt.plot(train_sensitivity, label='Training Sensitivity')
plt.plot(validation_sensitivity, label='Validation Sensitivity')

plt.title('Sensitivity Curve', fontsize=14)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Sensitivity', fontsize=12)
plt.xticks(range(len(train_sensitivity)))
plt.legend(loc='upper right', fontsize=12)
plt.tight_layout()
plt.show()

### **Precision Curve**

In [ ]:
train_precision = performance_metrics_resnet['train_precision']
validation_precision = performance_metrics_resnet['val_precision']

plt.figure(figsize=(10, 5))
plt.plot(train_precision, label='Training Precision')
plt.plot(validation_precision, label='Validation Precision')

plt.title('Precision Curve', fontsize=14)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Precision', fontsize=12)
plt.xticks(range(len(train_precision)))
plt.legend(loc='upper right', fontsize=12)
plt.tight_layout()
plt.show()

### **Confusion Matrix**

In [ ]:
model_cnn.load_state_dict(best_model_resnet)
model_cnn.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in dataloaders['val']:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model_cnn(images)
        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Compute confusion matrix
cm = confusion_matrix(all_labels, all_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=list(classDict.keys()))

plt.figure(figsize=(8, 6))
disp.plot(cmap='Blues', values_format='d', ax=plt.gca())
plt.title("Confusion Matrix (Validation Set)", fontsize=14)
plt.xlabel("Predicted Label", fontsize=12)
plt.ylabel("True Label", fontsize=12)
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)
plt.tight_layout()
plt.show()

# **Testing**

In [ ]:

def test(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0
    y_preds, y_trues = [], []

    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc="Testing"):
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss    = criterion(outputs, labels)
            total_loss += loss.item() * images.size(0)

            preds = torch.argmax(outputs, dim=1)
            y_preds.extend(preds.cpu().numpy())
            y_trues.extend(labels.cpu().numpy())

    test_loss = total_loss / len(dataloader.dataset)
    test_f1 = f1_score(y_trues, y_preds, average='macro', zero_division=0)
    test_prec = precision_score(y_trues, y_preds, average='macro', zero_division=0)
    test_rec = recall_score(y_trues, y_preds, average='macro', zero_division=0)
    test_acc = accuracy_score(y_trues, y_preds)

    print(f"Test Loss: {test_loss:.4f} | F1: {test_f1:.4f} | "
          f"Precision: {test_prec:.4f} | Recall: {test_rec:.4f} | Acc: {test_acc:.4f}")

    return {
        'loss': test_loss,
        'f1': test_f1,
        'precision': test_prec,
        'recall': test_rec,
        'accuracy': test_acc
    }


In [ ]:
# CNN
model_cnn.load_state_dict(best_model_cnn)
test_metrics_cnn = test(model_cnn, test_loader, calc_loss_cnn, device)

# ResNet
model_resnet.load_state_dict(best_model_resnet)
test_metrics_resnet = test(model_resnet, test_loader, calc_loss, device)

# UNet
model_unet.load_state_dict(best_model_unet)
test_metrics_unet = test(model_unet, test_loader, calc_loss, device)

# Swin
model_swin.load_state_dict(best_model_swin)
test_metrics_swin = test(model_swin, test_loader, calc_loss, device)
